# LLM-Based Term Extraction (Zero-Shot)

This notebook demonstrates a zero-shot term extraction approach using Large Language Models:
- Uses Groq API for term extraction
- Processes sentences in batches for efficiency
- No training required - relies on LLM's general knowledge

## Setup and Imports

In [18]:
import pandas as pd
import json
import os
import re
from tqdm import tqdm

print("Libraries imported")

Libraries imported


## Data Loading and Processing

In [19]:
# Load training and dev data
train_data = json.load(open('../data/taskA/train.json', 'r', encoding='utf-8'))
test_data = json.load(open('../data/taskA/test.json', 'r', encoding='utf-8'))

print(f"Training docs: {len(train_data)}")
print(f"Dev docs: {len(test_data)}")
print(f"\nExample sentence:")
print(f"  Text: {test_data[list(test_data.keys())[0]]['text']}")
print(f"  Terms: {test_data[list(test_data.keys())[0]]['terms']}")

Training docs: 540
Dev docs: 567

Example sentence:
  Text: Gut microbiota disturbances in hospitalized older adults with malnutrition and clinical outcomes.  OBJECTIVE: Malnutrition is one of the most threatening conditions in geriatric  populations. The gut microbiota has an important role in the host's metabolic and muscular health: however, its interplay with disease-related malnutrition is not well understood. We aimed to identify the association of malnutrition with the gut microbiota and predict clinical outcomes in hospitalized acutely ill older adults. METHODS: We performed a secondary longitudinal analysis in 108 geriatric patients from a prospective cohort evaluated at admission and 72 h of hospitalization. We collected clinical, demographic, nutritional, and 16S rRNA gene-sequenced gut microbiota data. Microbiota diversity, overall composition, and differential abundance were calculated and compared between patients with and without malnutrition. Microbiota features associa

## Evaluation Metrics

Using the official evaluation metrics from the competition.

In [20]:
def micro_f1_score(gold_standard, system_output):
    """
    Evaluates performance using Precision, Recall, and F1 score 
    based on individual term matching (micro-average).
    """
    total_true_positives = 0
    total_false_positives = 0
    total_false_negatives = 0
    
    for gold, system in zip(gold_standard, system_output):
        gold_set = set(gold)
        system_set = set(system)
        
        true_positives = len(gold_set.intersection(system_set))
        false_positives = len(system_set - gold_set)
        false_negatives = len(gold_set - system_set)
        
        total_true_positives += true_positives
        total_false_positives += false_positives
        total_false_negatives += false_negatives
    
    precision = total_true_positives / (total_true_positives + total_false_positives) if (total_true_positives + total_false_positives) > 0 else 0
    recall = total_true_positives / (total_true_positives + total_false_negatives) if (total_true_positives + total_false_negatives) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    return precision, recall, f1, total_true_positives, total_false_positives, total_false_negatives


def type_f1_score(gold_standard, system_output):
    """
    Evaluates performance using Type Precision, Type Recall, and Type F1 score
    based on the set of unique terms extracted at least once across the entire dataset.
    """
    all_gold_terms = set()
    for item_terms in gold_standard:
        all_gold_terms.update(item_terms)
    
    all_system_terms = set()
    for item_terms in system_output:
        all_system_terms.update(item_terms)
    
    type_true_positives = len(all_gold_terms.intersection(all_system_terms))
    type_false_positives = len(all_system_terms - all_gold_terms)
    type_false_negatives = len(all_gold_terms - all_system_terms)
    
    type_precision = type_true_positives / (type_true_positives + type_false_positives) if (type_true_positives + type_false_positives) > 0 else 0
    type_recall = type_true_positives / (type_true_positives + type_false_negatives) if (type_true_positives + type_false_negatives) > 0 else 0
    type_f1 = 2 * (type_precision * type_recall) / (type_precision + type_recall) if (type_precision + type_recall) > 0 else 0
    
    return type_precision, type_recall, type_f1


print("Evaluation functions defined")

Evaluation functions defined


## Initialize LLM Model

In [21]:
import google.generativeai as genai
from openai import OpenAI
from dotenv import load_dotenv

groq_api_key = input("Enter your Groq API key: ")
os.environ['GROQ_API_KEY'] = groq_api_key

# Load environment variables
load_dotenv()

client = OpenAI(
    api_key=os.environ.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

# The full list of available models can be found at https://console.groq.com/docs/rate-limits
#model_name = "openai/gpt-oss-20b"
model_name = "openai/gpt-oss-120b"

response = client.responses.create(
    input="Hi, how are you?",
    model=model_name,
)

print(f"{model_name} response: {response.output_text}")

openai/gpt-oss-120b response: Hello! I'm doing great, thank you for asking. How can I help you today?


In [22]:
# API reference: https://console.groq.com/docs/api-reference#chat
response = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": "Hi, how are you?",
        }
    ],
    model=model_name,
)

print(response)

ChatCompletion(id='chatcmpl-a0f257b1-2f77-4087-b460-90aab8d684a0', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='I’m doing well, thanks! How can I assist you today?', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning='We need to follow system instructions. System says we must be concise and limit output to max 150 words. Also we must not mention system messages. So respond concisely, under 150 words.'))], created=1794418156, model='openai/gpt-oss-120b', object='chat.completion', service_tier='on_demand', system_fingerprint='fp_fe269835c7', usage=CompletionUsage(completion_tokens=65, prompt_tokens=77, total_tokens=142, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=42, rejected_prediction_tokens=None), prompt_tokens_details=None, queue_time=0.048595398, prompt_time=0.003560002, completion_time=0.13509283

In [23]:
print(response.choices[0].message.reasoning)
print(response.choices[0].message.content)

We need to follow system instructions. System says we must be concise and limit output to max 150 words. Also we must not mention system messages. So respond concisely, under 150 words.
I’m doing well, thanks! How can I assist you today?


In [24]:
print(response.usage.prompt_tokens)
print(response.usage.completion_tokens)
print(response.usage.total_tokens)

77
65
142


## Configure Zero-Shot Prompt

In [25]:
# Configure batch size for efficient processing
batch_size = 5

system_prompt = f"""You are an automatic term extraction agent. You will receive a list of texts as input.
Your role is to extract terms relevant to the gut-brain interplay field of research, including gut microbiota, mental health, and neurodegenerative diseases. Output a list of terms for each sentence.

Strictly adhere to the Example Output Format:

Example Output Format:
Document 1: [term1; term2; term3; term4]
Document 2: [term5; term6]
Document 3: []
Document 4: []
Document 5: [term7]

Instructions:
* Extract single-word and multi-word terms, including domain-relevant organism names such as Akkermansia muciniphila
* Extract domain-relevant lexical units even if they are not listed in biomedical dictionaries or ontologies
* Extract complete lexical units, including relevant modifiers, rather than their nested component terms
* Split coordinated expressions into separate terms, restoring shared words where needed: "cognitive or motor dysfunction" yields "cognitive dysfunction" and "motor dysfunction"
* Extract only terms relevant to the gut-brain interplay, ignoring other domains
* If a document contains no terms, output an empty list for that document
* You must output {batch_size} lists of terms, one for each document, without additional commentary
"""

system_prompt_single_doc = f"""You are an automatic term extraction agent. You will receive a biomedical text as input.
Your role is to extract terms relevant to the gut-brain interplay field of research, including gut microbiota, mental health, and neurodegenerative diseases. Output a list of terms.

Strictly adhere to the Example Output Format:

Example Output Format:
Terms: [term1; term2; term3; term4]

Instructions:
* Extract single-word and multi-word terms, including domain-relevant organism names such as Akkermansia muciniphila
* Extract domain-relevant lexical units even if they are not listed in biomedical dictionaries or ontologies
* Extract complete lexical units, including relevant modifiers, rather than their nested component terms
* Split coordinated expressions into separate terms, restoring shared words where needed: "cognitive or motor dysfunction" yields "cognitive dysfunction" and "motor dysfunction"
* Extract only terms relevant to the gut-brain interplay, ignoring other domains
* If a document contains no terms, output an empty list for that document
* You must output a single list of terms without additional commentary
"""

if batch_size == 1:
  system_prompt = system_prompt_single_doc

print(f"Zero-shot prompt configured (batch size: {batch_size})")

Zero-shot prompt configured (batch size: 5)


## Extract Terms Using LLM

In [26]:
test_data = json.load(open('../data/taskA/test.json', 'r', encoding='utf-8'))
MAX_DOCUMENTS = 20  # Quick first run; use None to evaluate the full test set.
if MAX_DOCUMENTS is not None:
    test_data = dict(list(test_data.items())[:MAX_DOCUMENTS])
print(f"Selected {len(test_data)} test documents")

Selected 20 test documents


In [27]:
import time
rpm_limit = 30 # requests per minute limit for the API (check https://console.groq.com/docs/rate-limits)

# Process documents in batches with LLM
print(f"Processing {len(test_data)} documents in batches of {batch_size}...")

raw_responses = []
response_list = []
user_prompt = ""

i=0
for id, content in tqdm(test_data.items(), desc="Processing documents..."):
    document = content['text']
    i += 1
    
    # Build prompt until batch size is reached
    if i % batch_size == 0:
        user_prompt += f"Document {i}:\n {document}\n\n"
        
        # Send batch to LLM
        response = client.chat.completions.create(
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            model=model_name,
            reasoning_effort="low",
            max_completion_tokens=4096,
            temperature=0.0  
        )
        
        raw_responses.append(response)
        response_list.append(response.choices[0].message.content)
        
        user_prompt = "" # Reset user prompt for next batch
        time.sleep(60 / rpm_limit)  # Respect API rate limit
    else:
        user_prompt += f"Document {i}:\n {document}\n\n"

# Process remaining documents (last batch)
if user_prompt:
    user_prompt = user_prompt.rstrip()
    response = client.chat.completions.create(
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        model=model_name,
        reasoning_effort="low",
        max_completion_tokens=4096,
        temperature=0.0  
    )
    raw_responses.append(response)
    response_list.append(response.choices[0].message.content)


print(f"Received {len(response_list)} batch responses from LLM")

Processing 20 documents in batches of 5...


Processing documents...: 100%|██████████| 20/20 [00:30<00:00,  1.51s/it]

Received 4 batch responses from LLM


### Analyze LLM outputs

In [28]:
for batch_num, response in enumerate(raw_responses, start=1):
    choice = response.choices[0]

    print(f"\n--- Batch {batch_num} ---")
    print("Model:", response.model)
    print("Finish reason:", choice.finish_reason)
    print("Content:\n", choice.message.content)
    print("Usage:", response.usage)

    if not (choice.message.content or "").strip():
        print("Full message:", choice.message.model_dump())


--- Batch 1 ---
Model: openai/gpt-oss-120b
Finish reason: stop
Content:
 Document 1: [gut microbiota; malnutrition; α-diversity; Lachnospiraceae NK4A136 group; Subdoligranulum; Faecalibacterium prausnitzii; Corynebacterium; Ruminococcaceae Incertae Sedis; Fusobacterium; clinical outcomes]  
Document 2: [gut microbiota]  
Document 3: [gut microbiota dysbiosis; Enterobacteriaceae; Ruminococcaceae; Lachnospiraceae; gut-lung axis; gut-brain axis; acute ischemic stroke; COVID-19 infection; functional outcomes]  
Document 4: [gut-brain axis; probiotics; postbiotics; anxiety; depression; mental health; gut microbiota; mood regulation]  
Document 5: [gut-brain axis; gut microbiota; microbiome-derived metabolites; arketamine; chronic restrain stress; depression; synaptic proteins; prefrontal cortex; L-leucine; N-acetyl-l-glutamine; L-threonine amide]
Usage: CompletionUsage(completion_tokens=713, prompt_tokens=2217, total_tokens=2930, completion_tokens_details=CompletionTokensDetails(accepted_p

In [29]:
for response in response_list:
    print(f"Response:\n{response}\n{'-'*40}")

Response:
Document 1: [gut microbiota; malnutrition; α-diversity; Lachnospiraceae NK4A136 group; Subdoligranulum; Faecalibacterium prausnitzii; Corynebacterium; Ruminococcaceae Incertae Sedis; Fusobacterium; clinical outcomes]  
Document 2: [gut microbiota]  
Document 3: [gut microbiota dysbiosis; Enterobacteriaceae; Ruminococcaceae; Lachnospiraceae; gut-lung axis; gut-brain axis; acute ischemic stroke; COVID-19 infection; functional outcomes]  
Document 4: [gut-brain axis; probiotics; postbiotics; anxiety; depression; mental health; gut microbiota; mood regulation]  
Document 5: [gut-brain axis; gut microbiota; microbiome-derived metabolites; arketamine; chronic restrain stress; depression; synaptic proteins; prefrontal cortex; L-leucine; N-acetyl-l-glutamine; L-threonine amide]
----------------------------------------
Response:
Document 6: [iron therapy; intestinal microbiome; gut microbiota composition; pathogenic bacteria; bifidobacteria; lactobacilli; dysbiosis; inflammatory bowel

## Parse LLM Responses

In [30]:
# Parse by document number so a missing/malformed line cannot shift later predictions.
print("Parsing LLM responses...")

llm_preds = [[] for _ in test_data]
if batch_size == 1:
  idx = 0
  for id, content in test_data.items():
      llm_preds[idx] = [term.strip().lower() for term in response_list[idx].split(';') if term.strip()]
      idx += 1
else:
    for batch_index, response in enumerate(response_list):
        first_id = batch_index * batch_size + 1
        expected_ids = set(range(first_id, min(first_id + batch_size, len(test_data) + 1)))
        seen_ids = set()
        for line in response.splitlines():
            match = re.fullmatch(r'\s*Document\s+(\d+):\s*\[(.*?)\]\s*', line)
            if not match:
                continue
            doc_num = int(match.group(1))
            if doc_num not in expected_ids or doc_num in seen_ids:
                print(f"Warning: ignoring unexpected/duplicate Document {doc_num} in batch {batch_index + 1}")
                continue
            terms = [term.strip().lower() for term in match.group(2).split(';') if term.strip()]
            llm_preds[doc_num - 1] = terms
            seen_ids.add(doc_num)
        missing_ids = expected_ids - seen_ids
        if missing_ids:
            print(f"Warning: missing/malformed output for Documents {sorted(missing_ids)}; using empty lists.")

print(f"Parsed {len(llm_preds)} predictions")


Parsing LLM responses...
Parsed 20 predictions


## Save Predictions to File

In [31]:
def save_predictions(predictions, documents, output_path):
    """Save predictions in competition format."""
    output = {}
    idx = 0
    for id, content in documents.items():
        output[id] = {
            'text': content['text'],
            'terms': predictions[idx]
        }
        idx += 1
    
    os.makedirs(os.path.dirname(output_path) or '.', exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(output, f, ensure_ascii=False, indent=2)
    print(f"Saved {len(predictions)} predictions to {output_path}")

save_predictions(llm_preds, test_data, f'predictions/subtaskA_{model_name.split("/")[-1].lower()}_preds.json')

Saved 20 predictions to predictions/subtaskA_gpt-oss-120b_preds.json


## Evaluate Performance

In [32]:
# Prepare gold standard and predictions for evaluation
test_gold = [content['terms'] for _, content in test_data.items()]

# Calculate metrics
precision, recall, f1, tp, fp, fn = micro_f1_score(test_gold, llm_preds)
type_precision, type_recall, type_f1 = type_f1_score(test_gold, llm_preds)

print("\n" + "="*60)
print("LLM ZERO-SHOT BASELINE RESULTS")
print("="*60)
print("\nMicro-averaged Metrics:")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1 Score:  {f1:.4f}")
print(f"  TP={tp}, FP={fp}, FN={fn}")

print("\nType-level Metrics:")
print(f"  Type Precision: {type_precision:.4f}")
print(f"  Type Recall:    {type_recall:.4f}")
print(f"  Type F1 Score:  {type_f1:.4f}")
print("="*60)


LLM ZERO-SHOT BASELINE RESULTS

Micro-averaged Metrics:
  Precision: 0.5385
  Recall:    0.1972
  F1 Score:  0.2887
  TP=126, FP=108, FN=513

Type-level Metrics:
  Type Precision: 0.4681
  Type Recall:    0.1709
  Type F1 Score:  0.2504


In [33]:
## Example Predictions

In [34]:
predictions = json.load(open(f'predictions/subtaskA_{model_name.split("/")[-1].lower()}_preds.json', 'r', encoding='utf-8'))

# Show example predictions
print("Example Predictions:\n")

count = 0

for id, content in test_data.items():
    if len(content['terms']) > 0 and count < 5:
        print(f"Document ID: {id}")
        print(f"Text: {content['text'][:100]}...")
        print(f"Gold terms: {content['terms'][:5]}")
        print(f"LLM predictions: {predictions[id]['terms'][:5]}")
        
        correct = set(content['terms']) & set(predictions[id]['terms'])
        missed = set(content['terms']) - set(predictions[id]['terms'])
        wrong = set(predictions[id]['terms']) - set(content['terms'])
        
        print(f"Correct: {len(correct)}")
        print(f"Missed: {len(missed)}")
        print(f"Wrong: {len(wrong)}")
        print("-"*80)
        print()
        
        count += 1


Example Predictions:

Document ID: mh_2
Text: Gut microbiota disturbances in hospitalized older adults with malnutrition and clinical outcomes.  O...
Gold terms: ['Gut microbiota disturbances', 'malnutrition', 'clinical outcomes', 'Malnutrition', 'gut microbiota']
LLM predictions: ['gut microbiota', 'malnutrition', 'α-diversity', 'lachnospiraceae nk4a136 group', 'subdoligranulum']
Correct: 4
Missed: 29
Wrong: 6
--------------------------------------------------------------------------------

Document ID: mh_3
Text: Preface: Special issue: "Ion channels and genetic epilepsy".  This preface introduces the Journal of...
Gold terms: ['Ion channels', 'genetic epilepsy', 'Epilepsy', 'disease', 'recurrent seizures']
LLM predictions: ['gut microbiota']
Correct: 1
Missed: 29
Wrong: 0
--------------------------------------------------------------------------------

Document ID: mh_5
Text: Association between gut microbiota dysbiosis and poor functional outcomes in acute ischemic stroke p...
Gold